In [ ]:
# compare modeled and observed carbon use efficiency (NPP/GPP) at GEM sites

In [ ]:
import numpy as np
#
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
#
## this next library contains some helper functions for reshaping FATES variables into easier to work with dimensions
import ctsm_py.fates_xarray_funcs as fa       
#
import textwrap
from cmcrameri import cm
import cartopy
import cartopy.crs as ccrs
from cartopy.io.shapereader import Reader
import pandas as pd


In [ ]:
plt.rcParams['figure.constrained_layout.use'] = True
plt.rcParams['figure.autolayout'] = False
plt.rcParams['font.size'] = 14
##plt.rcParams['text.usetex'] = True
plt.rcParams['figure.dpi'] = 300


In [ ]:
# read in GEM data from Huanyuan Zhang's v20260317 'GEM_biomass_GPP' spreadsheet
# units for NPP columns are MgC/ha/year
cols = [
    'Plot_code', 'Latitude', 'Longitude','LAI', 'CUE','NPP','GPP'
]

df = pd.read_csv(
    '/pscratch/sd/j/jkowalcz/e3sm_scratch/pm-cpu/Benchmarks/GEM/from_HZ_GEM_biomass_GPP_v20260317.csv',
    sep=',',
    usecols=cols,
    na_values=['Not measured', 'Not mentioned', 'Bad data']
)

# Cast numeric columns to float
numeric_cols = [c for c in cols if c != 'Plot_code']
df[numeric_cols] = df[numeric_cols].astype(float)

In [ ]:
print(df.columns.tolist())

In [ ]:
tag='hydro_comp_bmort_kmax_tune_sapflow_kmax3_5xHR_fixSLA_vcmax40.55_fixHR_branch'

path='/pscratch/sd/j/jkowalcz/e3sm_scratch/pm-cpu/SummaryFiles/'

file=tag+'.elm.h0.cat.nc'
d0=xr.open_dataset(path+file)


In [ ]:
lai=d0.TLAI.mean(dim='time') ## ask Huanyuan if LAI should exclude DBH < 10 cm too?

In [ ]:
fates_npp=fa.scpf_to_scls_by_pft(d0.FATES_NPP_SZPF,d0)
fates_gpp=fa.scpf_to_scls_by_pft(d0.FATES_GPP_SZPF,d0)

In [ ]:
cue=fates_npp.isel(fates_levscls=slice(2,14)).sum(dim='fates_levscls').sum(dim='fates_levpft').mean(dim='time')/fates_gpp.isel(fates_levscls=slice(2,14)).sum(dim='fates_levscls').sum(dim='fates_levpft').mean(dim='time')

In [ ]:
## calcuate FATES GPP, NPP for DBH > 10!

In [ ]:
nplants_szpf=fa.scpf_to_scls_by_pft(d0.FATES_NPLANT_SZPF,d0)

In [ ]:
coexist=(nplants_szpf.isel(fates_levpft=0).isel(fates_levscls=slice(2,14)).sum(dim='fates_levscls').mean(dim="time"))/(nplants_szpf.sum(dim="fates_levpft").isel(fates_levscls=slice(2,14)).sum(dim='fates_levscls').mean(dim="time"))


In [ ]:
# code by Claude AI
# --- Extract modeled values at each site ---
def extract_modeled_at_sites(df, da):
    """Extract nearest-grid-cell modeled values for each site in df."""
    values = []
    df['Longitude'] = df['Longitude'] % 360
    for _, row in df.iterrows():
        val = da.sel(lat=row['Latitude'], lon=row['Longitude'], method='nearest').item()
        values.append(val)
    return values

In [ ]:
# --- Bar plot function ---
def plot_comparison_bars(df, obs_col, mod_col, ylabel, title, ax):
    df_plot = df.dropna(subset=[obs_col])  # filter to non-NaN rows for this variable
    sites = df_plot['Plot_code']
    x = np.arange(len(sites))
    width = 0.35

    ax.bar(x - width/2, df_plot[obs_col], width, label='GEM', color='steelblue')
    ax.bar(x + width/2, df_plot[mod_col], width, label='FATES', color='darkorange')

    ax.set_xticks(x)
    ax.set_xticklabels(sites, rotation=45, ha='right', fontsize=12)
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.legend(loc='upper right',frameon=False)

In [ ]:
def plot_coexistence_ratio(df, obs_col, ylabel, title, ax):
    df_plot = df.dropna(subset=[obs_col])  # filter to non-NaN rows for this variable
    sites = df_plot['Plot_code']
    x = np.arange(len(sites))
    width = 0.35
    plt.axhline(y=0.5,linestyle='dashed',color="black")
    ax.bar(x, df_plot[obs_col], width, color='grey')
    ax.set_xticks(x)
    ax.set_xticklabels(sites, rotation=45, ha='right', fontsize=12)
    ax.set_ylabel(ylabel)
    ax.set_title(title)

In [ ]:
# convert kg C / m2/s to MgC/ha/year
cf=315576000.

In [ ]:
df['modeled_LAI'] = extract_modeled_at_sites(df, lai)
df['modeled_CUE'] = extract_modeled_at_sites(df, cue)
df['modeled_GPP'] = extract_modeled_at_sites(df, cf*gpp_all)
df['modeled_NPP'] = extract_modeled_at_sites(df, cf*npp_all)
df['modeled_early_frac']= extract_modeled_at_sites(df, coexist)

In [ ]:
# --- Plot ---
fig, axes = plt.subplots(1, 1, figsize=(7, 5))

plot_comparison_bars(df, 'CUE', 'modeled_CUE',
                     ylabel='CUE',
                     title='Carbon use efficiency',
                    ax=axes)

plt.tight_layout()
plt.show()

In [ ]:
df.modeled_CUE[0:9].values,df.modeled_CUE[14:16].values

In [ ]:
fates_cue=np.concatenate([df.modeled_CUE[0:9].values,df.modeled_CUE[14:16].values])

In [ ]:
np.mean(fates_cue)

In [ ]:
np.std(fates_cue)

In [ ]:
gem_cue=df.CUE.values

In [ ]:
np.nanmean(gem_cue)

In [ ]:
np.nanstd(gem_cue)

In [ ]:
fig, axes = plt.subplots(1, 1, figsize=(7, 5))

plot_coexistence_ratio(df, 'modeled_early_frac',
                     ylabel='fraction NPLANTS early PFT',
                     title='PFT coexistence at GEM sites (DBH > 10 cm)',
                    ax=axes)

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 1, figsize=(7, 5))
plot_comparison_bars(df, 'GPP', 'modeled_GPP',
                     ylabel='GPP (MgC/ha/year)',
                     title='GEM and FATES GPP',
                     ax=axes)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 1, figsize=(7, 5))
plot_comparison_bars(df, 'NPP', 'modeled_NPP',
                     ylabel='NPP (MgC/ha/year)',
                     title='GEM and FATES NPP',
                     ax=axes)
plt.tight_layout()
plt.show()

In [ ]:
def plot_LAI(df, obs_col, mod_col, ylabel, title, ax):
    df_plot = df.dropna(subset=[obs_col])  # filter to non-NaN rows for this variable
    sites = df_plot['Plot_code']
    x = np.arange(len(sites))
    width = 0.35

    ax.bar(x - width/2, df_plot[obs_col], width, label='GEM', color='steelblue')
    ax.bar(x + width/2, df_plot[mod_col], width, label='FATES', color='darkorange')

    ax.set_xticks(x)
    ax.set_xticklabels(sites, rotation=45, ha='right', fontsize=12)
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.legend(loc='lower left',frameon=True)

In [ ]:
fig, axes = plt.subplots(1, 1, figsize=(7, 5))
plot_LAI(df, 'LAI', 'modeled_LAI',
                     ylabel='LAI (m² m⁻²)',
                     title='Leaf area index',
                     ax=axes)
plt.tight_layout()
plt.show()

In [ ]:
np.nanmean(df['LAI'])

In [ ]:
np.nanstd(df['LAI'])